In [107]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, mutual_info_score, mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
import numpy as np

In [108]:
data = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

In [109]:
#!wget $data

In [110]:
df = pd.read_csv("course_lead_scoring_2026.csv")

In [111]:
features = ['annual_income', 'lead_score', 'interaction_count', 'number_of_courses_viewed', 'lead_source', 'industry', 'employment_status', 'location']
target = 'converted'

Check if the missing values are presented in the features.
If there are missing values:
- For categorical features, replace them with 'NA'
- For numerical features, replace with with 0.0

In [112]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [113]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [114]:
df['annual_income'] = df['annual_income'].fillna(0.0)
df['lead_score'] = df['lead_score'].fillna(0.0)

In [115]:
df['lead_source'] = df['lead_source'].fillna('NA')
df['industry'] = df['industry'].fillna('NA')
df['employment_status'] = df['employment_status'].fillna('NA')
df['location'] = df['location'].fillna('NA')

What is the most frequent observation (mode) for the column industry?

In [116]:
df.industry.value_counts(normalize=True)

industry
technology       0.2346
retail           0.1850
healthcare       0.1680
finance          0.1440
education        0.1278
manufacturing    0.0926
NA               0.0480
Name: proportion, dtype: float64

Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?
- interaction_count and lead_score
- number_of_courses_viewed and lead_score
- number_of_courses_viewed and interaction_count
- annual_income and interaction_count

Only consider the pairs above when answering this question.

In [117]:
numerical_features = ['annual_income', 'lead_score', 'interaction_count', 'number_of_courses_viewed']
df[numerical_features].corr()

,annual_income,lead_score,interaction_count,number_of_courses_viewed
annual_income,1.000000,0.229496,0.122842,0.161300
lead_score,0.229496,1.000000,0.915746,0.757204
interaction_count,0.122842,0.915746,1.000000,0.721609
number_of_courses_viewed,0.161300,0.757204,0.721609,1.000000


In [118]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

# reset the indexes of the dataframes
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# separate the target variable from the train/val/test sets
y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

# delete the target column from the train/val/test sets
del df_train[target]
del df_val[target]
del df_test[target]

Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.

Round the scores to 2 decimals using round(score, 2).

Which of these variables has the biggest mutual information score?
- industry
- location
- lead_source
- employment_status

In [119]:
categorical = ['industry', 'lead_source', 'employment_status', 'location']
numerical = ['annual_income', 'lead_score', 'interaction_count', 'number_of_courses_viewed']

In [120]:
for cat in categorical:
    score = round(mutual_info_score(df_train[cat], y_train), 2)
    print(f"{cat}: {score}")

industry: 0.0
lead_source: 0.03
employment_status: 0.02
location: 0.0


Now let's train a logistic regression.

Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.

Fit the model on the training dataset.

To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
- model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

In [124]:
def calc_accuracy(df_train, df_val, cat_features, num_features, y_train, y_val, C_value = 1.0) -> float:
    # Convert each category into a binary vector (a series of 0s and 1s).
    # Each category becomes a new column with a 1 or 0 indicating the presence of that category.
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    # one-hot encode the categorical variables
    X_train_cat_encoded = encoder.fit_transform(df_train[cat_features])
    X_train = np.concatenate([X_train_cat_encoded, df_train[num_features].to_numpy()], axis=1)

    model = LogisticRegression(solver='liblinear', C=C_value, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    # process the validation set in the same way you processed the training set
    X_val_cat_encoded = encoder.transform(df_val[cat_features])
    X_val = np.concatenate([X_val_cat_encoded, df_val[num_features].to_numpy()], axis=1)

    # calculate the accuracy on the validation dataset and round it to 2 decimal digits
    y_pred = model.predict(X_val)

    # Calculate accuracy on validation set
    accuracy = accuracy_score(y_val, y_pred)

    return accuracy

In [125]:
acc = calc_accuracy(df_train, df_val, categorical, numerical, y_train, y_val)
print(f'Accuracy: {round(acc, 2)}')

Accuracy: 0.65


Let's find the least useful feature using the feature elimination technique.

Train a model using the same features and parameters as in Q4 (without rounding).

Now exclude each feature from this set and train a model without it. Record the accuracy for each model.

For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?
- 'lead_source'
- 'number_of_courses_viewed'
- 'interaction_count'

In [126]:
acc_ls = calc_accuracy(
    df_train, df_val,
    [feature for feature in categorical if feature != 'lead_source'],
    numerical, y_train, y_val
)
acc_ic = calc_accuracy(
    df_train, df_val,
    categorical,
    [feature for feature in numerical if feature != 'interaction_count'],
    y_train, y_val
)
acc_ncv = calc_accuracy(
    df_train, df_val,
    categorical,
    [feature for feature in numerical if feature != 'number_of_courses_viewed'],
    y_train, y_val
)

print(f"Difference without lead_source: {acc - acc_ls:.4f}")
print(f"Difference without interaction_count: {acc - acc_ic:.4f}")
print(f"Difference without number_of_courses_viewed: {acc - acc_ncv:.4f}")

Difference without lead_source: 0.0030
Difference without interaction_count: 0.0440
Difference without number_of_courses_viewed: 0.0020


Now let's train a regularized logistic regression.

Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].

Train models using all the features as in Q4.

Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these C leads to the best accuracy on the validation set?

In [127]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

In [129]:
for c in C_values:
    acc = calc_accuracy(df_train, df_val, categorical, numerical, y_train, y_val, C_value=c)
    print(f"C={c}: Accuracy={round(acc, 3)}")

C=1e-06: Accuracy=0.598
C=1e-05: Accuracy=0.598
C=0.0001: Accuracy=0.613
C=0.001: Accuracy=0.645
